# pix2pix 가상 염색 (가짜 짝 데이터) 노트북

커널 **Python (blood)**, 위에서부터 순서대로 `Shift + Enter`. 실험 계획 문서의 **1번 실험**입니다.

```
[준비] TXL 염색 조각 ──규칙──→ 가짜 비염색   ⇒  (가짜 비염색, 진짜 염색) 짝
[학습] pix2pix: 가짜 비염색 → 염색 (짝을 보고 직접 배움)
[평가] 진짜 비염색(RedTell) → pix2pix → YOLO(conf 0.7) → RBC 개수  ← CycleGAN·하한과 같은 규칙
```

| 단계 | 하는 일 | 시간 |
| --- | --- | --- |
| 0 | 폴더 이동 | 바로 |
| 1 | 준비 (한 번만): 결과 폴더 정리, 가짜 짝 만들기, 가짜 비염색 확인 | 수 분 |
| 2 | pix2pix 학습 (짧은 테스트 → 본 학습) | 본 학습 수 시간 |
| 3 | epoch 고르기 (val 9장) | 수 분 |
| 4 | 최종 채점 (test 91장, 한 번만) | 1~2분 |
| 5 | 눈으로 확인 + 비교표 | 바로 |

**저장 위치** (새 정리 구조)
- 가짜 짝: `04. Prep/5_paired_synthetic/rbc_only_v2/` (A 가짜 비염색 · B 염색 · AB 붙인 것)
- 학습 가중치: `05. Train/runs/pix2pix/pix2pix_rbc/`
- 가상 염색 영상: `06. Generate/pix2pix/ep<숫자>/{val,test}/`
- 채점 결과: `07. Evaluate/3_staining/pix2pix/ep<숫자>_{val,test}/`

## 0. 폴더 이동 (노트북을 열 때마다 먼저)

In [1]:
import torch, os, sys
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
os.chdir(r'D:\05. Blood\05. Train\code')
print('스크립트 실행 파이썬:', sys.executable)
REPO = r"D:/05. Blood/05. Train/external/cyclegan"

GPU: True NVIDIA GeForce RTX 5060 Ti
스크립트 실행 파이썬: C:\Users\KonYang\anaconda3\envs\blood\python.exe


## 1. 준비 (처음 한 번만)

### 1-1. 결과 폴더 정리 (지우는 것 없음, 옮기기만)
`07. Evaluate/` 를 단계별(1_detection · 2_no_staining · 3_staining · 4_cell_mask)로, `06. Generate/` 를 모델별로 나눕니다.
`run_train.ipynb`, `run_staining.ipynb` 안의 경로도 새 위치로 바뀝니다 (바꾸기 전 사본은 `_backup_before_organize/`).

**실행 전에 그 두 노트북 탭은 닫아 주세요.** 먼저 `--dry_run` 줄로 무엇이 옮겨질지 보고, 괜찮으면 아래 줄을 실행합니다.

In [2]:
!{sys.executable} 00_organize_folders.py --train "D:/05. Blood/05. Train" --dry_run

[보기] results\cellmask_cyclegan_ep15_test → results\4_cell_mask\cellmask_cyclegan_ep15_test
[보기] results\cellmask_none_test → results\4_cell_mask\cellmask_none_test
[보기] results\eval0_stained_test → results\1_detection\eval0_stained_test
[보기] results\eval0_yolo_conf0.25 → results\1_detection\eval0_yolo_conf0.25
[보기] results\eval0_yolo_conf0.4 → results\1_detection\eval0_yolo_conf0.4
[보기] results\eval0_yolo_conf0.5 → results\1_detection\eval0_yolo_conf0.5
[보기] results\eval0_yolo_conf0.6 → results\1_detection\eval0_yolo_conf0.6
[보기] results\eval0_yolo_conf0.7 → results\1_detection\eval0_yolo_conf0.7
[보기] results\eval0_yolo_conf0.8 → results\1_detection\eval0_yolo_conf0.8
[보기] results\eval2_unstained_no_staining → results\2_no_staining\eval2_unstained_no_staining
[보기] results\eval2_unstained_no_staining_conf0.7 → results\2_no_staining\eval2_unstained_no_staining_conf0.7
[보기] results\stain_cyclegan_ep10_val → results\3_staining\cyclegan\ep10_val
[보기] results\stain_cyclegan_ep15_test → resul

In [3]:
!{sys.executable} 00_organize_folders.py --train "D:/05. Blood/05. Train"

results\cellmask_cyclegan_ep15_test → results\4_cell_mask\cellmask_cyclegan_ep15_test
results\cellmask_none_test → results\4_cell_mask\cellmask_none_test
results\eval0_stained_test → results\1_detection\eval0_stained_test
results\eval0_yolo_conf0.25 → results\1_detection\eval0_yolo_conf0.25
results\eval0_yolo_conf0.4 → results\1_detection\eval0_yolo_conf0.4
results\eval0_yolo_conf0.5 → results\1_detection\eval0_yolo_conf0.5
results\eval0_yolo_conf0.6 → results\1_detection\eval0_yolo_conf0.6
results\eval0_yolo_conf0.7 → results\1_detection\eval0_yolo_conf0.7
results\eval0_yolo_conf0.8 → results\1_detection\eval0_yolo_conf0.8
results\eval2_unstained_no_staining → results\2_no_staining\eval2_unstained_no_staining
results\eval2_unstained_no_staining_conf0.7 → results\2_no_staining\eval2_unstained_no_staining_conf0.7
results\stain_cyclegan_ep10_val → results\3_staining\cyclegan\ep10_val
results\stain_cyclegan_ep15_test → results\3_staining\cyclegan\ep15_test
results\stain_cyclegan_ep15_val 

### 1-2. 가짜 짝 만들기 (규칙 v2)
TXL-PBC 염색 조각(`rbc_only/trainB` 1,378개)마다 가짜 비염색을 2개씩 만들어 학습 짝 2,756쌍을 만듭니다. 규칙은 `04. Prep/code/06_make_fake_pairs.py` 맨 위 설명에 있습니다.
`--check`: TXL 검증 영상 252장 전체도 가짜 비염색으로 바꿔 둡니다 (1-4에서 사용).

- **v1** (첫 버전): 세포가 "선 그림"처럼 나오고 겹친 세포가 한 덩어리가 됨 → YOLO recall 0.15로 실제(0.72)와 너무 달라 사용 안 함. 결과는 `5_paired_synthetic/rbc_only/`에 그대로 둠
- **v2**: 실제 비염색의 **입체감**(가장자리 한쪽 어둡고 반대쪽 밝음) 추가, 붙은 세포를 하나씩 나눠 세포마다 경계선. 결과는 `5_paired_synthetic/rbc_only_v2/`


In [ ]:
!{sys.executable} "D:/05. Blood/04. Prep/code/06_make_fake_pairs.py" --prep "D:/05. Blood/04. Prep" --rule v2 --variants 2 --check

### 1-3. 눈으로 확인 — 왼쪽 염색(진짜) / 오른쪽 가짜 비염색, 맨 오른쪽 열은 비교용 실제 비염색

In [ ]:
import matplotlib.pyplot as plt, numpy as np, cv2, glob
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False  # 한글 제목용 (윈도우 기본 글꼴)
rd = lambda p, f=1: cv2.imdecode(np.fromfile(p, np.uint8), f)
prev = rd(r"D:/05. Blood/04. Prep/5_paired_synthetic/rbc_only_v2/preview.png")
real = [rd(p, 0) for p in sorted(glob.glob(r"D:/05. Blood/04. Prep/3_virtual_staining/rbc_only/trainA/dse_*.png"))[:8]]
fig, ax = plt.subplots(1, 2, figsize=(8, 12), gridspec_kw={'width_ratios': [2, 1]})
ax[0].imshow(cv2.cvtColor(prev, cv2.COLOR_BGR2RGB)); ax[0].set_title('염색 | 가짜 비염색')
ax[1].imshow(np.vstack(real), cmap='gray'); ax[1].set_title('실제 비염색 (학습용)')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

### 1-4. 가짜 비염색이 실제 비염색만큼 "어려운지" 확인
같은 YOLO로 TXL 검증 영상을 채점합니다. 원래 염색 영상은 RBC recall 약 0.95, **실제 비염색(RedTell)은 약 0.72**입니다.
가짜 비염색의 recall이 0.72 근처면 실제와 비슷하게 어렵다는 뜻이고, 훨씬 낮으면 실제보다 더 낯선 영상이라는 뜻입니다 (pix2pix가 진짜 비염색에서 덜 통할 수 있음).

**주의**: TXL 검증 영상은 출처마다 세포 밀도가 달라서, 출처별 표(`by_source`)도 같이 봅니다. BCCD는 세포가 빽빽하게 겹쳐 있어 가짜 비염색으로 바꾸면 특히 어렵고, 실제 비염색(RedTell)은 세포가 떨어져 있습니다. 시험용 14장 기준 v2는 전체 recall 약 0.17이었지만 세포가 드문 BCDD 영상은 1.0이었습니다.


In [ ]:
!{sys.executable} 12_eval_compare.py --prep "D:/05. Blood/04. Prep" --images "D:/05. Blood/04. Prep/5_paired_synthetic/check_txl_val_v2/images" --labels "D:/05. Blood/04. Prep/5_paired_synthetic/check_txl_val_v2/labels" --yolo "D:/05. Blood/05. Train/runs/yolo/yolo11s/weights/best.pt" --conf 0.7 --out "D:/05. Blood/07. Evaluate/3_staining/pix2pix/fake_unstained_check_txl_val_v2"
import pandas as pd
d = r"D:/05. Blood/07. Evaluate/3_staining/pix2pix/fake_unstained_check_txl_val_v2"
display(pd.read_csv(d + "/summary.csv")[["n_images", "RBC_MAE", "RBC_precision", "RBC_recall", "RBC_F1"]])
display(pd.read_csv(d + "/by_source.csv")[["source", "n_images", "RBC_MAE", "RBC_precision", "RBC_recall", "RBC_F1"]])

## 2. pix2pix 학습

설정: 공식 pix2pix(조건부 GAN + L1 손실 100, U-Net 생성기, 70×70 PatchGAN) 그대로 + 아래만 바꿈
- `--input_nc 1`: 비염색은 흑백 1채널 (CycleGAN 1차의 실수 반복 안 함)
- `--netG unet_128`: 128×128 조각에 맞는 U-Net
- `--norm instance`: 조각 단위로 변환할 때 결과가 안정적이도록 (공식 기본은 batch)
- **100 epoch** (50 + 학습률 감소 50), **5 epoch마다 저장** → 3번에서 val로 고름

### 2-1. 짧은 테스트 (1~2분). `End of epoch 1 / 1`이 나오면 성공. 출력의 `time:` 값으로 본 학습 시간 예상

In [ ]:
!{sys.executable} "D:/05. Blood/05. Train/external/cyclegan/train.py" --dataroot "D:/05. Blood/04. Prep/5_paired_synthetic/rbc_only_v2/AB" --name pix2pix_smoke --model pix2pix --direction AtoB --input_nc 1 --output_nc 3 --netG unet_128 --norm instance --checkpoints_dir "D:/05. Blood/05. Train/runs/pix2pix" --preprocess none --load_size 128 --crop_size 128 --n_epochs 1 --n_epochs_decay 0 --max_dataset_size 200 --num_threads 0 --print_freq 50 --display_freq 100000

### 2-2. 본 학습
- 1 epoch = 2,756 iteration. 예상 시간 ≈ 2-1의 `time` × 2,756 × 100 초
- 노트북 창을 닫지 마세요. 끊기면 같은 명령 끝에 `--continue_train --epoch_count <다음 epoch>`
- 중간 결과 그림: `runs/pix2pix/pix2pix_rbc/web/index.html`

In [ ]:
!{sys.executable} "D:/05. Blood/05. Train/external/cyclegan/train.py" --dataroot "D:/05. Blood/04. Prep/5_paired_synthetic/rbc_only_v2/AB" --name pix2pix_rbc --model pix2pix --direction AtoB --input_nc 1 --output_nc 3 --netG unet_128 --norm instance --checkpoints_dir "D:/05. Blood/05. Train/runs/pix2pix" --preprocess none --load_size 128 --crop_size 128 --n_epochs 50 --n_epochs_decay 50 --save_epoch_freq 5 --num_threads 0 --print_freq 1000 --display_freq 5000 --update_html_freq 5000

## 3. epoch 고르기 (val 9장으로만)
**규칙: val RBC F1이 가장 높은 epoch.** 실제 비염색(진짜) val로 고릅니다. 가짜 짝의 val로 고르지 않습니다.
변환은 학습 때와 같은 128 조각 단위(`22_apply_generator.py`, 겹쳐서 붙임).

In [ ]:
EPOCHS = list(range(5, 101, 5))
for e in EPOCHS:
    !{sys.executable} 22_apply_generator.py --repo "D:/05. Blood/05. Train/external/cyclegan" --ckpt "D:/05. Blood/05. Train/runs/pix2pix/pix2pix_rbc" --epoch {e} --images "D:/05. Blood/04. Prep/4_eval_unstained/redtell_dse_val/images" --out "D:/05. Blood/06. Generate/pix2pix/ep{e}/val"
    !{sys.executable} 12_eval_compare.py --prep "D:/05. Blood/04. Prep" --images "D:/05. Blood/06. Generate/pix2pix/ep{e}/val" --labels "D:/05. Blood/04. Prep/4_eval_unstained/redtell_dse_val/labels" --yolo "D:/05. Blood/05. Train/runs/yolo/yolo11s/weights/best.pt" --conf 0.7 --out "D:/05. Blood/07. Evaluate/3_staining/pix2pix/ep{e}_val"

In [ ]:
import pandas as pd
cols = ["epoch", "RBC_MAE", "RBC_precision", "RBC_recall", "RBC_F1"]
rows = [pd.read_csv(rf"D:/05. Blood/07. Evaluate/2_no_staining/stain_none_val/summary.csv").assign(epoch="가상 염색 없음")]
rows += [pd.read_csv(rf"D:/05. Blood/07. Evaluate/3_staining/pix2pix/ep{e}_val/summary.csv").assign(epoch=e) for e in EPOCHS]
tab = pd.concat(rows)[cols]
display(tab)
best = tab[tab.epoch != "가상 염색 없음"].sort_values("RBC_F1", ascending=False).iloc[0]
print("val RBC F1 최고 epoch:", best.epoch)

## 4. 최종 채점 (test 91장) — `BEST_EPOCH`를 3번 결과로 바꾸고 **한 번만**

In [ ]:
BEST_EPOCH = 100   # ← 3번 결과로 바꾸기
!{sys.executable} 22_apply_generator.py --repo "D:/05. Blood/05. Train/external/cyclegan" --ckpt "D:/05. Blood/05. Train/runs/pix2pix/pix2pix_rbc" --epoch {BEST_EPOCH} --images "D:/05. Blood/04. Prep/4_eval_unstained/redtell_dse/images" --out "D:/05. Blood/06. Generate/pix2pix/ep{BEST_EPOCH}/test"
!{sys.executable} 12_eval_compare.py --prep "D:/05. Blood/04. Prep" --images "D:/05. Blood/06. Generate/pix2pix/ep{BEST_EPOCH}/test" --labels "D:/05. Blood/04. Prep/4_eval_unstained/redtell_dse/labels" --yolo "D:/05. Blood/05. Train/runs/yolo/yolo11s/weights/best.pt" --conf 0.7 --out "D:/05. Blood/07. Evaluate/3_staining/pix2pix/ep{BEST_EPOCH}_test"

## 5. 눈으로 확인 + 비교표
초록 선 = 실제 세포 위치(정답 윤곽). 세포가 없는 곳에 세포가 생기거나(환각), 있는 세포가 사라졌는지 봅니다.

In [ ]:
import matplotlib.pyplot as plt, numpy as np, cv2, os
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False  # 한글 제목용 (윈도우 기본 글꼴)
rd = lambda p, f=1: cv2.imdecode(np.fromfile(p, np.uint8), f)
CG = 15   # CycleGAN 1차에서 고른 epoch
names = ["dse_100.png", "dse_120.png", "dse_150.png"]
fig, ax = plt.subplots(len(names), 3, figsize=(13, 4.4 * len(names)))
for i, n in enumerate(names):
    gt = rd(rf"D:/05. Blood/04. Prep/4_eval_unstained/redtell_dse/masks/{n}", -1)
    edge = np.zeros(gt.shape, np.uint8)
    for v in np.unique(gt)[1:]:
        c, _ = cv2.findContours((gt == v).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE); cv2.drawContours(edge, c, -1, 255, 2)
    ims = [cv2.cvtColor(rd(rf"D:/05. Blood/04. Prep/4_eval_unstained/redtell_dse/images/{n}", 0), cv2.COLOR_GRAY2RGB),
           cv2.cvtColor(rd(rf"D:/05. Blood/06. Generate/cyclegan/ep{CG}/test/{n}"), cv2.COLOR_BGR2RGB),
           cv2.cvtColor(rd(rf"D:/05. Blood/06. Generate/pix2pix/ep{BEST_EPOCH}/test/{n}"), cv2.COLOR_BGR2RGB)]
    for j, (im, t) in enumerate(zip(ims, ["입력 (비염색)", f"CycleGAN ep{CG}", f"pix2pix ep{BEST_EPOCH}"])):
        im = im.copy(); im[edge > 0] = (0, 200, 0)
        ax[i, j].imshow(im); ax[i, j].set_title(f"{n} · {t}"); ax[i, j].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
import pandas as pd
cols = ["model", "RBC_MAE", "RBC_precision", "RBC_recall", "RBC_F1"]
t = pd.concat([pd.read_csv(rf"D:/05. Blood/07. Evaluate/2_no_staining/stain_none_test/summary.csv").assign(model="비염색 그대로 (하한)"),
               pd.read_csv(rf"D:/05. Blood/07. Evaluate/3_staining/cyclegan/ep15_test/summary.csv").assign(model="CycleGAN 1차 ep15"),
               pd.read_csv(rf"D:/05. Blood/07. Evaluate/3_staining/pix2pix/ep{BEST_EPOCH}_test/summary.csv").assign(model=f"pix2pix ep{BEST_EPOCH}")])
display(t[cols])